In [14]:
# Copyright 2024 Daniel Franzen and Jan Disselhoff
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import json
from unsloth import FastLanguageModel
from diskcache import Cache

from arc_loader import ArcDataset
from model_tools import load_unsloth_4bit
from inference_tools import inference_run
from selection import EvalTool
from arc_downloader import download_arc_data

In [15]:
# input paths
base_model = 'Llama-3.2-3B-Instruct-merged'
arc_data_path = os.path.join('input', 'arc-prize-2024')  # as on kaggle arc prize 2024
download_arc_data(arc_data_path)

# output paths
output_path = 'output_evaluation_Llama-arc_without_ttt'
save_model_path = os.path.join('finetuned_models', base_model)
inference_cache = os.path.join(output_path, 'inference_cache')
submission_file = os.path.join(output_path, 'submission.json')

# load evaluation dataset
arc_eval_set = ArcDataset.load_from_json_dl('../../../../dataset', n=1, sizes=[3], seed=42, shuffle=False)

load arc dataset: 100%|██████████| 301/301 [00:10<00:00, 28.22it/s]


In [16]:
arc_eval_set.solutions['arc-a8d7556c00']

[[[9, 0, 0, 9, 2, 2],
  [9, 9, 9, 9, 2, 2],
  [0, 0, 9, 0, 9, 9],
  [9, 9, 9, 0, 2, 2],
  [9, 2, 2, 9, 2, 2],
  [9, 2, 2, 0, 9, 9]]]

In [17]:
# load model
model, tokenizer = load_unsloth_4bit(save_model_path)

# set formatting options
fmt_opts = dict(
    preprompt='ABCDEFGHJKLMNPQRSTUVWXYZabcdefghjklmnpqrstuvwxyz',
    query_beg='I',
    reply_beg='\n+/-=O',
    reply_end='\n' + tokenizer.eos_token,
    lines_sep='\n',
    max_tokens=128000,
)

`rope_scaling`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=8192


==((====))==  Unsloth 2025.4.7: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.
   \\   /|    NVIDIA TITAN Xp. Num GPUs = 1. Max memory: 11.897 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.1. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`rope_scaling`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=8192
`rope_scaling`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=8192


In [18]:
# run inference
FastLanguageModel.for_inference(model)
infer_aug_opts = dict(tp='all', rt='all', perm=True, shfl_ex=True, seed=10000)
infer_dataset = arc_eval_set.repeat(2).augment(**infer_aug_opts)
model_cache = Cache(inference_cache).memoize(typed=True, ignore=set(['model_tok', 'guess']))
eval_tool = EvalTool(n_guesses=2)

In [20]:
inference_results = inference_run(
    model_tok=(model, tokenizer),
    fmt_opts=fmt_opts,
    dataset=infer_dataset,
    min_prob=0.1,
    aug_score_opts=infer_aug_opts,
    callback=eval_tool.process_result,
    cache=model_cache,
)

 * task 'arc-007bbfb700_0': 0 candidates, correct solution not found
   max_gen_prob:    0.00% (  0.00/ 28.00)
   max_aug_prob:    0.00% (  0.00/ 28.00)
   min_aug_prob:    0.00% (  0.00/ 28.00)
   sum_aug_prob:    0.00% (  0.00/ 28.00)
   mul_aug_prob:    0.00% (  0.00/ 28.00)
   mul_all_prob:    0.00% (  0.00/ 28.00)
   correct_found:   0.00% (  0.00/ 28.00)

 * task 'arc-00d62c1b00_0': 0 candidates, correct solution not found
   max_gen_prob:    0.00% (  0.00/ 29.00)
   max_aug_prob:    0.00% (  0.00/ 29.00)
   min_aug_prob:    0.00% (  0.00/ 29.00)
   sum_aug_prob:    0.00% (  0.00/ 29.00)
   mul_aug_prob:    0.00% (  0.00/ 29.00)
   mul_all_prob:    0.00% (  0.00/ 29.00)
   correct_found:   0.00% (  0.00/ 29.00)

 * task 'arc-017c7c7b00_0': 0 candidates, correct solution not found
   max_gen_prob:    0.00% (  0.00/ 30.00)
   max_aug_prob:    0.00% (  0.00/ 30.00)
   min_aug_prob:    0.00% (  0.00/ 30.00)
   sum_aug_prob:    0.00% (  0.00/ 30.00)
   mul_aug_prob:    0.00% (  0.00/ 

inference:   5%|▍         | 14/300 [00:00<00:08, 35.27it/s]

   max_aug_prob:    0.00% (  0.00/ 36.00)
   min_aug_prob:    0.00% (  0.00/ 36.00)
   sum_aug_prob:    0.00% (  0.00/ 36.00)
   mul_aug_prob:    0.00% (  0.00/ 36.00)
   mul_all_prob:    0.00% (  0.00/ 36.00)
   correct_found:   0.00% (  0.00/ 36.00)

 * task 'arc-09629e4f00_0': 0 candidates, correct solution not found
   max_gen_prob:    0.00% (  0.00/ 37.00)
   max_aug_prob:    0.00% (  0.00/ 37.00)
   min_aug_prob:    0.00% (  0.00/ 37.00)
   sum_aug_prob:    0.00% (  0.00/ 37.00)
   mul_aug_prob:    0.00% (  0.00/ 37.00)
   mul_all_prob:    0.00% (  0.00/ 37.00)
   correct_found:   0.00% (  0.00/ 37.00)

 * task 'arc-0a938d7900_0': 0 candidates, correct solution not found
   max_gen_prob:    0.00% (  0.00/ 38.00)
   max_aug_prob:    0.00% (  0.00/ 38.00)
   min_aug_prob:    0.00% (  0.00/ 38.00)
   sum_aug_prob:    0.00% (  0.00/ 38.00)
   mul_aug_prob:    0.00% (  0.00/ 38.00)
   mul_all_prob:    0.00% (  0.00/ 38.00)
   correct_found:   0.00% (  0.00/ 38.00)

 in: 443 out:  3/ 3

inference:   6%|▌         | 18/300 [00:00<00:09, 28.80it/s]

 * task 'arc-11852cab00_0': 1 candidates, correct solution not found
   max_gen_prob:    0.00% (  0.00/ 43.00)
   max_aug_prob:    0.00% (  0.00/ 43.00)
   min_aug_prob:    0.00% (  0.00/ 43.00)
   sum_aug_prob:    0.00% (  0.00/ 43.00)
   mul_aug_prob:    0.00% (  0.00/ 43.00)
   mul_all_prob:    0.00% (  0.00/ 43.00)
   correct_found:   0.00% (  0.00/ 43.00)

 in: 273 out:  5/  7 >  2x1  bad_xy_size p=26% [arc-1190e5a700_0.perm8054217963.ex1-0-2.out0]
 in: 273 out:  5/  7 >  2x1  bad_xy_size p=34% [arc-1190e5a700_0.rt.rt.perm7569241830.ex2-1-0.out0]
 in: 273 out:  5/  7 >  2x1  bad_xy_size p=36% [arc-1190e5a700_0.tp.rt.rt.rt.perm7283410695.ex1-2-0.out0]
 in: 273 out:  5/  7 >  2x1  bad_xy_size p=23% [arc-1190e5a700_0.perm0679218345.ex0-2-1.out0]
 in: 273 out:  5/  7 >  2x1  bad_xy_size p=20% [arc-1190e5a700_0.tp.rt.perm6294715830.ex2-0-1.out0]
 in: 273 out:  5/  7 >  2x1  bad_xy_size p=17% [arc-1190e5a700_0.rt.rt.perm2153786490.ex1-0-2.out0]
 in: 273 out:  5/  7 >  2x1  bad_xy_size p

...

 * task 'arc-28e73c2000_0': 0 candidates, correct solution not found
   max_gen_prob:    1.33% (  1.00/ 75.00)
   max_aug_prob:    1.33% (  1.00/ 75.00)
   min_aug_prob:    1.33% (  1.00/ 75.00)
   sum_aug_prob:    1.33% (  1.00/ 75.00)
   mul_aug_prob:    1.33% (  1.00/ 75.00)
   mul_all_prob:    1.33% (  1.00/ 75.00)
   correct_found:   1.33% (  1.00/ 75.00)



inference:  16%|█▋        | 49/300 [08:22<1:27:42, 20.97s/it]

 * task 'arc-2962317100_0': 0 candidates, correct solution not found
   max_gen_prob:    1.32% (  1.00/ 76.00)
   max_aug_prob:    1.32% (  1.00/ 76.00)
   min_aug_prob:    1.32% (  1.00/ 76.00)
   sum_aug_prob:    1.32% (  1.00/ 76.00)
   mul_aug_prob:    1.32% (  1.00/ 76.00)
   mul_all_prob:    1.32% (  1.00/ 76.00)
   correct_found:   1.32% (  1.00/ 76.00)



inference:  16%|█▋        | 49/300 [08:33<43:52, 10.49s/it]  


KeyboardInterrupt: 